In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Define the 2D Rosenbrock function and its derivatives
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

def grad_rosenbrock(x):
    g1 = -2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2)
    g2 = 200 * (x[1] - x[0]**2)
    return np.array([g1, g2])

def hess_rosenbrock(x):
    h11 = 2 - 400 * x[1] + 1200 * x[0]**2
    h12 = -400 * x[0]
    h22 = 200
    return np.array([[h11, h12], [h12, h22]])

# 2. Armijo Backtracking Back-up
def armijo_backtrack(x, d, g, alpha_start=1.0, beta=0.5, epsilon=0.1):
    alpha = alpha_start
    f_x = rosenbrock(x)
    while rosenbrock(x + alpha * d) > f_x + epsilon * alpha * np.dot(g, d):
        alpha *= beta
        if alpha < 1e-12:
            break
    return alpha

# 3. Optimization Algorithms
def gradient_descent_exact(x0, c_tol=0.001, k_max=5000):
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    
    for k in range(k_max):
        g = grad_rosenbrock(x)
        if np.linalg.norm(g) <= c_tol:
            return x, np.linalg.norm(g), k, np.array(history)
        
        d = -g
        H = hess_rosenbrock(x)
        denom = np.dot(d, np.dot(H, d))
        
        if denom <= 0:
            alpha = armijo_backtrack(x, d, g)
        else:
            alpha = np.dot(g, g) / denom
            if rosenbrock(x + alpha * d) >= rosenbrock(x):
                alpha = armijo_backtrack(x, d, g)
                
        x += alpha * d
        history.append(x.copy())
        
    return x, np.linalg.norm(grad_rosenbrock(x)), k_max, np.array(history)

def newton_method(x0, c_tol=0.001, k_max=5000):
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    
    for k in range(k_max):
        g = grad_rosenbrock(x)
        if np.linalg.norm(g) <= c_tol:
            return x, np.linalg.norm(g), k, np.array(history)
        
        H = hess_rosenbrock(x)
        try:
            d = np.linalg.solve(H, -g)
        except np.linalg.LinAlgError:
            d = -g
            
        x += d
        history.append(x.copy())
        
    return x, np.linalg.norm(grad_rosenbrock(x)), k_max, np.array(history)

# 4. Run optimization and print report
initial_guesses = [
    [-1.2, 1.0],
    [-1.0, -1.0],
    [0.0, 0.0],
    [1.0, 0.0],
    [1.0, 1.2]
]

print("--- 5(a) OPTIMIZATION REPORT ---")
results = {}
for i, x0 in enumerate(initial_guesses):
    res_gd = gradient_descent_exact(x0)
    res_nt = newton_method(x0)
    results[i] = {'gd_hist': res_gd, 'nt_hist': res_nt}
    
    print(f"\nInitial Guess x0 = {x0}")
    print(f"  Gradient Descent: Sol={np.round(res_gd[0],4)}, Criterion={res_gd[1]:.6f}, Iters={res_gd[2]}")
    print(f"  Newton's Method:  Sol={np.round(res_nt[0],4)}, Criterion={res_nt[1]:.6f}, Iters={res_nt[2]}")

# 5. Generate Contour Plots
X1, X2 = np.meshgrid(np.linspace(-2.0, 2.0, 400), np.linspace(-1.5, 3.0, 400))
Z = (1 - X1)**2 + 100 * (X2 - X1**2)**2

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.contour(X1, X2, Z, levels=np.logspace(-1, 3, 20), cmap='jet', alpha=0.6)
for i, x0 in enumerate(initial_guesses):
    hist = results[i]['gd_hist'][3]
    plt.plot(hist[:, 0], hist[:, 1], '-o', markersize=3, label=f'x0={x0}')
plt.plot(1, 1, 'k*', markersize=15, label='Global Min (1,1)')
plt.title("Gradient Method with Exact Line Search")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend(loc='lower left', fontsize='small')

plt.subplot(1, 2, 2)
plt.contour(X1, X2, Z, levels=np.logspace(-1, 3, 20), cmap='jet', alpha=0.6)
for i, x0 in enumerate(initial_guesses):
    hist = results[i]['nt_hist'][3]
    plt.plot(hist[:, 0], hist[:, 1], '-o', markersize=4, label=f'x0={x0}')
plt.plot(1, 1, 'k*', markersize=15, label='Global Min (1,1)')
plt.title("Newton's Method")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend(loc='lower left', fontsize='small')

plt.tight_layout()
plt.show()
